## Binned Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML

class GlyphClassifier(nn.Module):
    def __init__(self, NUM_bins, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 256),
            nn.ReLU(),
            nn.Linear(256, NUM_bins)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  
        x = self.classifier(x)     
        return x


In [ ]:
# Getting the dataset
dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train")
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test')

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=5)


In [ ]:
import torch
import matplotlib.pyplot as plt
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
num_bins = 20
bin_centers = torch.linspace(2.5, 97.5, num_bins).to(device)  # Center of bins
model = GlyphClassifier(resolution=(128, 128), NUM_bins=num_bins).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
wandb.init(
    project="glyph-regression",
    name="exp-SimpleStar-128x128-20bins-BinnedRegression",
    config={
        "architecture": "CNN-Glyph",
        "epochs": 10,
        "batch_size": 64,
        "learning_rate": 0.0005,
        "loss_fn": "MSELoss",
        "optimizer": "Adam",
        "image_resolution": (128, 128),
        "regression": True,
        "num_bins": num_bins,
    }
)
wandb.watch(model, log="all", log_freq=10)
num_epochs = 20
losses = []
global_step = 0
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, values in train_loader:
        images = images.to(device)
        values = torch.tensor(values, dtype=torch.float32, device=device)
        optimizer.zero_grad()
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1) 
        predictions = torch.sum(probabilities * bin_centers, dim=1)
        loss = criterion(predictions, values)       
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        losses.append(loss.item())
        wandb.log({
            "train_loss_step": loss.item(),
            "global_step": global_step
        })
        if global_step % 100 == 0:
            print(f"Step {global_step}: Loss = {loss.item():.4f}")
        global_step += 1
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")

ML.plot_training_loss(losses, title="Training Loss over Steps", xlabel="Steps", ylabel="MSE Loss")

In [ ]:
import numpy as np
# Evaluation

model.eval()
predictions = []
ground_truths = []

with torch.no_grad():
    for images, values in test_loader:  
        images = images.to(device)
        values = torch.tensor(values, dtype=torch.float32, device=device)  
        logits = model(images)
        probabilities = F.softmax(logits, dim=1)
        batch_predictions = torch.sum(probabilities * bin_centers, dim=1)  
        predictions.extend(batch_predictions.cpu().numpy())
        ground_truths.extend(values.cpu().numpy())  
predictions = np.array(predictions)
ground_truths = np.array(ground_truths)
mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))
print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}")
wandb.log({
    "test_mse": mse,
    "test_mae": mae
})
wandb.finish()

### Validation for binned regression 

In [ ]:
dataset_file = 'data/simple-star-validation.zip'
validation_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split='test')
validation_loader = ML.create_loader(validation_dataset, batch_size=64, shuffle=False)

ML.visualize_loader(validation_loader,max_images=10,nrow=5)

In [ ]:
import pandas as pd

model.eval()
val_predictions = []
val_ground_truths = []
val_sample_ids = []

with torch.no_grad():
    for batch_idx, (images, values) in enumerate(validation_loader):
        images = images.to(device)
        values = torch.tensor(values, dtype=torch.float32, device=device)
        
        logits = model(images)
        probabilities = F.softmax(logits, dim=1)
        preds = torch.sum(probabilities * bin_centers, dim=1)
        
        val_predictions.extend(preds.cpu().numpy())
        val_ground_truths.extend(values.cpu().numpy())

        # Extract filenames (sample IDs) from the validation dataset
        start_idx = batch_idx * validation_loader.batch_size
        end_idx = start_idx + len(images)
        sample_ids = [
            validation_dataset.samples[i]["file"]
            for i in range(start_idx, end_idx)
        ]
        val_sample_ids.extend(sample_ids)


In [ ]:
df_val = pd.DataFrame({
    "sample_id": val_sample_ids,
    "ground_truth": val_ground_truths,
    "predicted_value": val_predictions
})
df_val['error'] = np.abs(df_val['ground_truth'] - df_val['predicted_value'])


print(df_val)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

val_mae = mean_absolute_error(df_val['ground_truth'], df_val['predicted_value'])
val_mse = mean_squared_error(df_val['ground_truth'], df_val['predicted_value'])

print(f"Validation MAE: {val_mae:.4f}")
print(f"Validation MSE: {val_mse:.4f}")

In [ ]:
# Hist of error

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(df_val['error'], bins=num_bins, color='steelblue', edgecolor='black')
plt.title("Distribution of Absolute Errors")
plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# Fetch N highest error
N = 10
top_errors = df_val.sort_values(by='error', ascending=False).head(N)
print(top_errors[['sample_id', 'ground_truth', 'predicted_value', 'error']])
